In [26]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
import pickle
import configparser
from sklearn.model_selection import train_test_split

In [28]:
sys.path.append('./tools')

In [29]:
MiM_path = Path(os.getcwd()).parent
DIR = sys.path.append(MiM_path/'data')
BASE = Path(MiM_path/'data')

# HRF

In [30]:
raw_HRF = Path(BASE/'raw-data/HRF - Quality')

In [31]:
with open(BASE/'labels/HRF.pkl','rb') as f:
    HRF = pickle.load(f)

In [34]:
HRF_x = HRF.filename
HRF_y = HRF.label

In [76]:
HRF_x_train, HRF_x_rest, HRF_y_train, HRF_y_rest = train_test_split(HRF_x,HRF_y,test_size = 0.5,stratify = HRF_y)

In [77]:
HRF_x_val, HRF_x_test, HRF_y_val, HRF_y_test = train_test_split(HRF_x_rest,HRF_y_rest,test_size = 0.3,stratify = HRF_y_rest)

In [84]:
HRF_config = configparser.ConfigParser()
HRF_config['split'] = {'type':'holdout',
                    'training':','.join(HRF_x_train.apply(lambda x: 'HRF - Quality/'+x.split('.')[0]).tolist()),
                    'validation':','.join(HRF_x_val.apply(lambda x: 'HRF - Quality/'+x.split('.')[0]).tolist()),
                    'test':','.join(HRF_x_test.apply(lambda x: 'HRF - Quality/'+x.split('.')[0]).tolist())}

with open(BASE/'splits/HRF.ini', 'w') as configfile:
    HRF_config.write(configfile)

# DDR

In [130]:
raw_DDR = Path(BASE/'raw-data/DDR-dataset/DR_grading')

In [131]:
DDR_train = pd.read_csv(raw_DDR/'train.txt',sep=' ', header = None)
DDR_val = pd.read_csv(raw_DDR/'valid.txt',sep=' ', header = None)
DDR_test = pd.read_csv(raw_DDR/'test.txt',sep=' ', header = None)

In [132]:
DDR_config = configparser.ConfigParser()
DDR_config['split'] = {'type':'holdout',
                    'training':','.join(DDR_train[0].apply(lambda x: 'DDR-dataset/'+x.split('.')[0]).tolist()),
                    'validation':','.join(DDR_val[0].apply(lambda x: 'DDR-dataset/'+x.split('.')[0]).tolist()),
                    'test':','.join(DDR_test[0].apply(lambda x: 'DDR-dataset/'+x.split('.')[0]).tolist())}

with open(BASE/'splits/DDR.ini', 'w') as configfile:
    DDR_config.write(configfile)

# Retinografía

In [108]:
with open(BASE/'labels/Retinografia.pkl','rb') as f:
    retinografia = pickle.load(f)

**Este dataset tiene 5 imágenes duplicadas. Tres de ellas con mismo label por lo que se elimina uno. Las otras dos imágenes repetidas tienen diferente label (buena calidad, regular calidad). En estas últimas se elimina el registro con label de buena calidad, asumiendo que la foto tiene algún defecto que merece ser reconocido por el modelo.**

In [109]:
retinografia = retinografia.drop(columns='file_path')
retinografia = retinografia.drop_duplicates()
duplicados = retinografia[retinografia.filename.duplicated()]
retinografia[retinografia.filename.isin(duplicados.filename)].sort_values(by='filename')

,filename,label
26,145.jpg,0
161,145.jpg,1
32,152.jpg,0
143,152.jpg,1


In [110]:
retinografia = retinografia[~((retinografia.filename.isin(duplicados.filename)) & (retinografia.label == 0))]

In [111]:
retinografia_x = retinografia.filename
retinografia_y = retinografia.label

In [112]:
retinografia_y.value_counts('%')

2    0.449704
0    0.319527
1    0.230769
Name: label, dtype: float64

In [113]:
ret_x_train,ret_x_rest,ret_y_train,ret_y_rest = train_test_split(retinografia_x,retinografia_y,test_size = 0.2,stratify = retinografia_y)

In [116]:
ret_x_val,ret_x_test,ret_y_val,ret_y_test = train_test_split(ret_x_rest,ret_y_rest,test_size = 0.4,stratify = ret_y_rest)

In [120]:
ret_config = configparser.ConfigParser()
ret_config['split'] = {'type':'holdout',
                    'training':','.join(ret_x_train.apply(lambda x: 'RETINOGRAFIA - Calidad/'+x.split('.')[0]).tolist()),
                    'validation':','.join(ret_x_val.apply(lambda x: 'RETINOGRAFIA - Calidad/'+x.split('.')[0]).tolist()),
                    'test':','.join(ret_x_test.apply(lambda x: 'RETINOGRAFIA - Calidad/'+x.split('.')[0]).tolist())}

with open(BASE/'splits/RETINOGRAFIA.ini', 'w') as configfile:
    ret_config.write(configfile)

# Kaggle

In [119]:
raw_kaggle = Path(BASE/'raw-data/Kaggle')

## Zhou

In [121]:
Zhou_train = pd.read_csv(raw_kaggle/'Zhou_train.csv',sep=',', header = 0)
Zhou_val = pd.read_csv(raw_kaggle/'Zhou_val.csv',sep=',', header = 0)
Zhou_test = pd.read_csv(raw_kaggle/'Zhou_test.csv',sep=',', header = 0)

In [122]:
Zhou_config = configparser.ConfigParser()
Zhou_config['split'] = {'type':'holdout',
                        'training':','.join(Zhou_train['image'].apply(lambda x: 'Kaggle/'+x.split('.')[0]).tolist()),
                        'validation':','.join(Zhou_val['image'].apply(lambda x: 'Kaggle/'+x.split('.')[0]).tolist()),
                        'test':','.join(Zhou_test['image'].apply(lambda x: 'Kaggle/'+x.split('.')[0]).tolist())}

with open(BASE/'splits/Zhou.ini', 'w') as configfile:
    Zhou_config.write(configfile)

## Fu

In [123]:
Fu_train = pd.read_csv(raw_kaggle/'Fu_train.csv',sep=',', header = 0)
Fu_test = pd.read_csv(raw_kaggle/'Fu_test.csv',sep=',', header = 0)

In [124]:
Fu_config = configparser.ConfigParser()
Fu_config['split'] = {'type':'holdout',
                    'training':','.join(Fu_train['image'].apply(lambda x: 'Kaggle/'+x.split('.')[0]).tolist()),
                    'validation':'',
                    'test':','.join(Fu_test['image'].apply(lambda x: 'Kaggle/'+x.split('.')[0]).tolist())}

with open(BASE/'splits/Fu.ini', 'w') as configfile:
    Fu_config.write(configfile)

# Deep Diabetic Retinopathy Image Dataset DeepDRiD

In [125]:
raw_DRiD = Path(BASE/'raw-data/Deep-Diabetic-Retinopathy-Image-Dataset-DeepDRiD-')

In [126]:
x = pd.read_csv(raw_DRiD/'regular_fundus_images/regular-fundus-training/regular-fundus-training.csv')
DRiD_train = x.loc[:,['image_id','Overall quality']]

In [127]:
x = pd.read_csv(raw_DRiD/'regular_fundus_images/regular-fundus-validation/regular-fundus-validation.csv')
DRiD_val = x.loc[:,['image_id','Overall quality']]

In [128]:
DRiD = pd.concat([DRiD_train,DRiD_val],ignore_index=True)
DRiD['Overall quality']=DRiD['Overall quality']+1
DRiD.loc[DRiD['Overall quality'] == 2,'Overall quality']=0

In [129]:
drid_config = configparser.ConfigParser()
drid_config['split'] = {'type':'holdout',
                    'training':'',
                    'validation':'',
                    'test':','.join(DRiD['image_id'].apply(lambda x: 'Deep-Diabetic-Retinopathy-Image-Dataset-DeepDRiD-/'+x.split('.')[0]).tolist())}

with open(BASE/'splits/DRID.ini', 'w') as configfile:
    drid_config.write(configfile)

# Global

In [135]:
global2_config = configparser.ConfigParser()
global2_config['split'] = {'type':'holdout',
                        'training':HRF_config['split']['training']+DDR_config['split']['training']+Zhou_config['split']['training'],
                        'validation':HRF_config['split']['validation']+DDR_config['split']['validation']+Zhou_config['split']['validation'],
                        'test':HRF_config['split']['test']+DDR_config['split']['test']+Zhou_config['split']['test']+drid_config['split']['test']
                        }

with open(BASE/'splits/global_binaria.ini', 'w') as configfile:
    global2_config.write(configfile)

In [136]:
global3_config = configparser.ConfigParser()
global3_config['split'] = {'type':'holdout',
                        'training':ret_config['split']['training']+Fu_config['split']['training'],
                        'validation':ret_config['split']['validation']+Fu_config['split']['validation'],
                        'test':ret_config['split']['test']+Fu_config['split']['test']
                        }

with open(BASE/'splits/global_3cat.ini', 'w') as configfile:
    global3_config.write(configfile)